# PantheraLAB 🐆 — IA real v4 · Detección + silueta + especie

**Mejora:** buscamos el cuerpo del perro con un modelo independiente de **segmentación semántica** (LR-ASPP/MobileNetV3, Torchvision), además de MegaDetector V6 Extra y SpeciesNet. Ahora la comparación muestra **la foto completa**, la **caja de MegaDetector**, una **silueta candidata** y los recortes usados para clasificar.

**Por qué:** en pruebas anteriores, MegaDetector encontró principalmente la cabeza de tu perro (78,6 %) y las rotaciones introdujeron falsos positivos de personas/vehículos. Por ello, **dejamos de girar la foto** y probamos una segunda IA especializada en segmentar categorías visuales.

**No prometemos detectar todo el cuerpo:** el modelo de segmentación está entrenado con categorías COCO/Pascal VOC (incluye perro, gato y algunos animales domésticos). **No tiene una clase específica de jaguar** y su máscara puede ser incompleta o errónea. La silueta no sustituye una identificación científica ni la caja de MegaDetector. Para felinos silvestres habrá que evaluar modelos generales o entrenamiento propio.

**Cómo usarlo:** en Google Colab, elegí **Entorno de ejecución → Ejecutar todas**, subí tu foto propia sin personas y esperá los informes. Es una prueba con una fotografía real, **NO video en vivo ni detección instalada en PantheraLIVE**. Los archivos se procesan en la sesión de Colab y no se envían a Render.

Fuentes: https://github.com/microsoft/MegaDetector · https://github.com/google/cameratrapai · https://docs.pytorch.org/vision/main/models/generated/torchvision.models.segmentation.lraspp_mobilenet_v3_large.html

## 1. Preparar los modelos

Instalá las dependencias en **la sesión temporal de Colab**. Torchvision ya acompaña normalmente a PyTorch en este entorno. Los pesos oficiales de los modelos se descargan la primera vez y pueden tardar varios minutos.

In [ ]:
%pip -q install PytorchWildlife speciesnet Pillow

## 2. Elegir la foto del perro u otro animal
Se ejecuta una única prueba sobre tu fotografía. Recomendación: JPG, PNG o WEBP de hasta 12 MB.

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image, ImageOps
from pathlib import Path
from io import BytesIO
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No se seleccionó una fotografía.")
name, payload = next(iter(uploaded.items()))
if len(payload)>12*1024*1024:
    raise ValueError("Fotografía mayor que 12 MB.")
photo = ImageOps.exif_transpose(Image.open(BytesIO(payload))).convert("RGB")
if photo.width*photo.height>25_000_000:
    raise ValueError("Fotografía mayor que 25 megapíxeles.")
folder = Path("/content/pantheralab_prueba_real")
folder.mkdir(parents=True,exist_ok=True)
file_path = folder/"fotograma.jpg"
photo.save(file_path,quality=92)
preview = photo.copy()
preview.thumbnail((820,820))
display(preview)
print("Fotografía preparada:",photo.width,"x",photo.height)

## 3. MegaDetector V6 Extra · Detección original

Se analiza **una sola vez la fotografía sin rotarla**, para evitar los falsos positivos de personas y vehículos observados en la prueba anterior. MegaDetector produce cajas de detección, pero no necesita encontrar el cuerpo completo para generar un candidato. Conservamos su puntuación original y no la confundimos con precisión científica.

In [ ]:
import torch
from PIL import ImageDraw
from PytorchWildlife.models import detection as pw_detection

device="cuda" if torch.cuda.is_available() else "cpu"
variant="MDV6-yolov10-e"
threshold=0.20
W,H=photo.size
print("MegaDetector:",variant,"| Procesador:",device)
model=pw_detection.MegaDetectorV6(device=device,pretrained=True,version=variant)
prediction=model.single_image_detection(str(file_path),det_conf_thres=threshold)
objects=prediction.get("detections")
names={0:"animal",1:"person",2:"vehicle"}
detections=[]
if objects is not None:
    for box,score,category in zip(objects.xyxy,objects.confidence,objects.class_id):
        xyxy=[int(round(float(v))) for v in box]
        x1,y1,x2,y2=xyxy
        if x2<=x1 or y2<=y1:
            continue
        detections.append({
            "class_name":names.get(int(category),"unknown"),
            "confidence":round(float(score),4),
            "bbox_xyxy":xyxy,
            "orientation_degrees":0,
            "box_fraction_of_image":round((x2-x1)*(y2-y1)/(W*H),5)
        })
detections.sort(key=lambda item:item["confidence"],reverse=True)
animals=[d for d in detections if d["class_name"]=="animal"]
people=[d for d in detections if d["class_name"]=="person"]
primary_detection=animals[0] if animals else None
person_review_candidates=[d for d in people if d["confidence"]>=0.40]
weak_person_candidates=[d for d in people if d["confidence"]<0.40]

annotated=photo.copy()
draw=ImageDraw.Draw(annotated)
for item in detections:
    box=item["bbox_xyxy"]
    color="#00dc9c" if item is primary_detection else "#ffc36b"
    draw.rectangle(box,outline=color,width=5 if item is primary_detection else 3)
    draw.text((box[0]+3,max(0,box[1]-18)),
              item["class_name"].upper()+" "+str(round(item["confidence"]*100))+"%",
              fill=color)
annotated.save(folder/"detecciones.jpg",quality=92)
overview=annotated.copy()
overview.thumbnail((920,920))
display(overview)
print("Animales:",len(animals),"| Posibles personas:",len(people),
      "| Posibles vehículos:",sum(d["class_name"]=="vehicle" for d in detections))
if primary_detection:
    print("Caja principal:",round(primary_detection["confidence"]*100,2),
          "% de confianza; cubre",round(primary_detection["box_fraction_of_image"]*100,2),
          "% de la foto. Esto NO mide cuánto del cuerpo está cubierto.")
else:
    print("⚠ No se encontró un candidato 'animal'; la segmentación posterior será independiente.")
if weak_person_candidates:
    print("⚠ Detecciones débiles de personas sin validar; revisá la foto antes de usarla.")
if person_review_candidates:
    print("⚠ Posible presencia humana. No usar esta foto para clasificación en este ensayo.")

## 4. Segmentación del cuerpo · Modelo LR-ASPP (Torchvision)

El modelo ligero **LR-ASPP/MobileNetV3** identifica píxeles correspondientes a algunas categorías (incluido **perro**) y puede producir una **silueta preliminar**, independientemente de la caja anterior. No se gira ni se altera la fotografía.

Solo aceptamos una máscara para comparación si su tamaño es plausible **y coincide espacialmente con la caja principal de MegaDetector**; si no, la mostramos como inconclusa. Los umbrales son experimentales y no demuestran exactitud científica.

**Limitación decisiva:** el modelo está entrenado para categorías comunes, **no identifica jaguares**. Con cámaras trampa será necesario otro modelo o entrenamiento/validación especializado.

In [ ]:
import numpy as np
import torch.nn.functional as F
from torchvision.models.segmentation import (
    lraspp_mobilenet_v3_large, LRASPP_MobileNet_V3_Large_Weights
)

segmentation_model="torchvision/lraspp_mobilenet_v3_large"
segmentation_status="not_run"
segmentation_candidates=[]
segmentation_selected=None
segmented_crop=None
segment_bbox=None
segmentation_overlay=None

if person_review_candidates:
    segmentation_status="person_review_required"
    print("Segmentación omitida: posible presencia de personas; revisar la foto.")
else:
    try:
        weights=LRASPP_MobileNet_V3_Large_Weights.DEFAULT
        categories=weights.meta["categories"]
        print("Cargando segmentador ligero. Categorías probadas: perro, gato y algunos animales domésticos.")
        segmenter=lraspp_mobilenet_v3_large(weights=weights).to(device).eval()
        tensor=weights.transforms()(photo).unsqueeze(0).to(device)
        with torch.inference_mode():
            raw=segmenter(tensor)["out"]
            probabilities=raw.softmax(dim=1)
            winning=probabilities.argmax(dim=1,keepdim=True)
        class_names=("dog","cat","bird","cow","horse","sheep")
        candidates=[]
        for class_name in class_names:
            if class_name not in categories:
                continue
            class_id=categories.index(class_name)
            # Upscale only the class index map (nearest), avoiding large GPU allocations.
            label_map=F.interpolate(
                winning.float(),size=(H,W),mode="nearest"
            )[0,0].round().to(torch.int64).cpu().numpy()
            mask=(label_map==class_id)
            area=int(mask.sum())
            if area<max(150,int(W*H*0.002)) or area>int(W*H*0.80):
                continue
            ys,xs=np.where(mask)
            x1,y1,x2,y2=int(xs.min()),int(ys.min()),int(xs.max()+1),int(ys.max()+1)
            overlap_fraction=0.0
            if primary_detection:
                ax,ay,bx,by=primary_detection["bbox_xyxy"]
                ax,ay,bx,by=max(ax,0),max(ay,0),min(bx,W),min(by,H)
                if bx>ax and by>ay:
                    overlap_fraction=float(mask[ay:by,ax:bx].mean())
            small_winner=(winning[0,0]==class_id)
            class_probability=(float(probabilities[0,class_id][small_winner].mean().item())
                               if bool(small_winner.any()) else 0.0)
            candidates.append({
                "class_name":class_name,
                "estimated_pixel_score":round(class_probability,4),
                "pixel_fraction_of_image":round(area/(W*H),5),
                "bbox_xyxy":[x1,y1,x2,y2],
                "overlap_with_megadetector":round(overlap_fraction,4),
                "_mask":mask
            })
        segmentation_candidates=[{
            k:v for k,v in candidate.items() if k!="_mask"
        } for candidate in candidates]
        plausible=[c for c in candidates
                   if primary_detection and c["overlap_with_megadetector"]>=0.10]
        plausible.sort(key=lambda c:(c["overlap_with_megadetector"],
                                     c["estimated_pixel_score"]),reverse=True)
        if plausible:
            chosen=plausible[0]
            segmentation_selected={k:v for k,v in chosen.items() if k!="_mask"}
            segment_bbox=chosen["bbox_xyxy"]
            mask_image=Image.fromarray((chosen["_mask"].astype(np.uint8)*255),mode="L")
            # Green-tinted mask is a genuine output of the segmenter, not an expanded box.
            tint=Image.new("RGB",photo.size,(18,209,146))
            blended=Image.blend(photo,tint,alpha=0.48)
            segmentation_overlay=Image.composite(blended,photo,mask_image)
            draw_seg=ImageDraw.Draw(segmentation_overlay)
            draw_seg.rectangle(segment_bbox,outline="#fff6be",width=5)
            draw_seg.text((segment_bbox[0]+5,max(0,segment_bbox[1]-19)),
                          chosen["class_name"].upper()+" · SEGMENTACIÓN EXPERIMENTAL",
                          fill="#fff6be")
            segmentation_overlay.save(folder/"segmentacion_animal.jpg",quality=92)
            segmented_crop=photo.crop(tuple(segment_bbox))
            segmentation_status="candidate"
            print("Silueta candidata:",chosen["class_name"],
                  "| puntuación media de píxeles:",round(chosen["estimated_pixel_score"]*100,1),"%",
                  "| superficie:",round(chosen["pixel_fraction_of_image"]*100,2),"% de la foto",
                  "| coincidencia con caja MD:",round(chosen["overlap_with_megadetector"]*100,1),"%")
            print("⚠ La máscara puede incluir fondo o partes incorrectas. No es una especie confirmada.")
        else:
            segmentation_status="no_associated_mask"
            print("No apareció una máscara plausible asociada con MegaDetector.")
            print("No se inventó ni amplió una caja: conservamos el resultado original.")
    except Exception as exc:
        segmentation_status="error"
        print("El segmentador no pudo ejecutarse:",type(exc).__name__,str(exc)[:240])
        print("MegaDetector y SpeciesNet aún pueden ejecutarse con sus recortes propios.")

## 5. SpeciesNet · Comparación de las regiones

Comparamos la clasificación del recorte exacto de MegaDetector, del recorte ampliado (solo contexto) y, **si existe una máscara válida**, de la caja derivada de segmentación.

Las tres entradas son distintas; las puntuaciones se comparan como **hipótesis del mismo modelo**, no como porcentajes de precisión. Si la segmentación no encuentra el cuerpo, no crearemos una caja ficticia.

El cuaderno mantiene el arreglo de KaggleHub de solo lectura para el modelo SpeciesNet y no modifica los archivos originales.

In [ ]:
from speciesnet.classifier import SpeciesNetClassifier
import kagglehub
import json
from pathlib import Path
from shutil import copy2

species_suggestions=[]
species_status="not_run"
species_note=""
species_views=[]
exact_crop=None
context_crop=None

def prepare_classifier_model():
    official_id="google/speciesnet/pyTorch/v4.0.3a/1"
    original=Path(kagglehub.model_download(official_id)).resolve()
    metadata=json.loads((original/"info.json").read_text(encoding="utf-8"))
    target=folder/"speciesnet_classifier_only"
    target.mkdir(parents=True,exist_ok=True)
    for key in ("classifier","classifier_labels"):
        if key not in metadata:
            raise RuntimeError("El modelo oficial no contiene "+key)
        relative=Path(metadata[key])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Ruta de pesos inesperada: "+str(relative))
        source_file=original/relative
        if not source_file.is_file():
            raise FileNotFoundError(str(source_file))
        target_file=target/relative
        target_file.parent.mkdir(parents=True,exist_ok=True)
        if not target_file.exists():
            try:
                target_file.symlink_to(source_file)
            except OSError:
                copy2(source_file,target_file)
    metadata["detector"]="unused_in_classifier_only_test.pt"
    (target/"info.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
    return target

if person_review_candidates:
    species_status="person_review_required"
    species_note="Hay una señal de persona suficientemente fuerte para requerir revisión manual. No se clasifica esta imagen."
    print("⚠ Revisá si la imagen contiene personas. Evitá subir imágenes de personas a Colab.")
elif weak_person_candidates:
    print("Aviso: el detector produjo señales débiles de persona en vistas giradas.",
          "No se interpretan automáticamente como presencia humana ni prueba de ausencia de personas.")
if species_status=="not_run" and primary_detection is None:
    species_status="no_animal"
    species_note="MegaDetector no encontró animales."
elif species_status=="not_run" and primary_detection is not None and primary_detection["box_fraction_of_image"]<0.01:
    species_status="insufficient_bbox"
    species_note="Caja principal demasiado pequeña para esta prueba de clasificación."
elif species_status=="not_run":
    x1,y1,x2,y2=primary_detection["bbox_xyxy"]
    exact_crop=photo.crop((x1,y1,x2,y2))
    w,h=x2-x1,y2-y1
    margin=0.65
    padded_bbox=(max(0,int(x1-w*margin)),max(0,int(y1-h*margin)),
                 min(W,int(x2+w*margin)),min(H,int(y2+h*margin)))
    context_crop=photo.crop(padded_bbox)

    try:
        print("Cargando clasificador SpeciesNet…")
        local_model_dir=prepare_classifier_model()
        classifier=SpeciesNetClassifier(model_name=str(local_model_dir),device=device)
        input_views=[("recuadro_del_detector",exact_crop),
                     ("recorte_con_contexto",context_crop)]
        if segmented_crop is not None:
            input_views.append(("caja_de_segmentacion",segmented_crop))
        for crop_name,crop_image in input_views:
            result=classifier.predict(crop_name+".jpg",classifier.preprocess(crop_image))
            classifications=result.get("classifications",{})
            suggestions=[{
                "label":str(label).rsplit(";",1)[-1].strip(),
                "raw_label":str(label),
                "score":round(float(score),4)
            } for label,score in zip(
                classifications.get("classes",[]),
                classifications.get("scores",[])
            )][:5]
            top=suggestions[0]["label"].lower() if suggestions else ""
            classification="not_identified" if top=="blank" else (
                "suggested" if suggestions else "no_prediction")
            species_views.append({
                "crop":crop_name,
                "bbox_xyxy":(
                    list(primary_detection["bbox_xyxy"]) if crop_name=="recuadro_del_detector"
                    else list(segment_bbox) if crop_name=="caja_de_segmentacion"
                    else list(padded_bbox)
                ),
                "region_provenance":(
                    "megadetector_box" if crop_name=="recuadro_del_detector"
                    else "lraspp_semantic_mask_bbox" if crop_name=="caja_de_segmentacion"
                    else "expanded_context_not_model_detection"
                ),
                "model_generated_bbox":crop_name in ("recuadro_del_detector","caja_de_segmentacion"),
                "status":classification,
                "suggestions":suggestions
            })
            print("\n"+crop_name.replace("_"," ").upper())
            for suggestion in suggestions[:3]:
                print("  ",suggestion["label"],round(suggestion["score"]*100,2),"%",
                      "— sugerencia sin validar")
        valid=[view for view in species_views if view["status"]=="suggested"]
        if valid:
            species_status="suggested"
            # El recorte exacto es el predeterminado; el ampliado solo añade contexto.
            preferred=next((view for view in valid if view["crop"]=="recuadro_del_detector"),valid[0])
            species_suggestions=preferred["suggestions"]
            species_note="Sugerencias reales; comparar las dos vistas y confirmar manualmente."
        else:
            species_status="not_identified"
            species_note="Ningún recorte produjo una especie reconocible."
    except Exception as exc:
        species_status="error"
        species_note=type(exc).__name__+": "+str(exc)[:230]
        print("No se pudo finalizar SpeciesNet:",species_note)

print("\nESTADO:",species_status,"·",species_note)

# Presentación explícita: foto original COMPLETA + detección real + recortes.
from PIL import ImageFont
panels=[("FOTO COMPLETA · original",photo),
        ("DETECCIONES REALES · cajas del modelo",annotated)]
if exact_crop is not None:
    panels.append(("RECORTE EXACTO · de MegaDetector",exact_crop))
if context_crop is not None:
    panels.append(("CONTEXTO EXTRA · NO es detección",context_crop))
if segmentation_overlay is not None:
    panels.append(("SEGMENTACIÓN · silueta candidata",segmentation_overlay))
if segmented_crop is not None:
    panels.append(("REGIÓN SEGMENTADA · NO es MegaDetector",segmented_crop))
panel_width=640
panel_pics=[]
for title,img in panels:
    preview=img.copy()
    preview.thumbnail((panel_width,660))
    background=Image.new("RGB",(panel_width,preview.height+46),"#101e2b")
    overlay=ImageDraw.Draw(background)
    overlay.text((12,14),title,fill="white")
    background.paste(preview,((panel_width-preview.width)//2,40))
    panel_pics.append(background)
comparison=Image.new("RGB",(panel_width,sum(i.height for i in panel_pics)+12*(len(panel_pics)-1)),"#102636")
offset=0
for image in panel_pics:
    comparison.paste(image,(0,offset))
    offset+=image.height+12
comparison.save(folder/"comparacion_visual_pantheralab.jpg",quality=90)
display(comparison)
print("Comparativa guardada: original, MegaDetector, recortes y máscara independiente.")
print("La segmentación es experimental; no reemplaza ni valida la caja de MegaDetector.")

## 6. Informe y comparación visual

Se descargan un **JSON** con las predicciones de los tres sistemas, una foto con las cajas originales de MegaDetector y una comparativa completa. Si la segmentación produjo una máscara plausible, se descarga también la imagen de esa máscara.

**Las predicciones no representan observaciones científicas validadas ni se registran automáticamente en PantheraLIVE.**

In [ ]:
from datetime import datetime, timezone
import json
summary={
    "platform":"PantheraLAB",
    "test":"Colab MegaDetector V6 Extra + LRASPP body segmentation + SpeciesNet",
    "timestamp":datetime.now(timezone.utc).isoformat(),
    "simulation":False,
    "scientifically_verified":False,
    "source":"authorized_image_uploaded_to_colab",
    "image_dimensions":{"width":W,"height":H},
    "megadetector":{
        "model":variant,"threshold":threshold,"orientations_tested":[0],
        "detections":detections,
        "weak_person_predictions":len(weak_person_candidates),
        "person_review_candidates":len(person_review_candidates),
        "person_gate_policy":"experimental: original 0deg >=0.35 or any view >=0.60; not a privacy guarantee",
        "primary_detection":primary_detection,
        "primary_box_selection":"highest confidence animal, original orientation only"
    },
    "segmentation":{
        "model":segmentation_model,
        "status":segmentation_status,
        "candidates":segmentation_candidates,
        "selected":segmentation_selected,
        "trained_for_jaguars":False,
        "mask_scientifically_validated":False
    },
    "speciesnet":{
        "status":species_status,"note":species_note,
        "suggestions":species_suggestions,"views":species_views,
        "context_bbox_is_model_detection":False,
        "segmentation_region_is_megadetector_bbox":False
    },
    "saved_in_pantheralive":False
}
report_file=folder/"resultado_pantheralab.json"
report_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print("RESULTADO:")
print(json.dumps(summary,ensure_ascii=False,indent=2)[:12000])
print("Estas probabilidades son puntuaciones de modelos, no identificaciones validadas.")

In [ ]:
from google.colab import files
files.download(str(report_file))
files.download(str(folder/"detecciones.jpg"))
files.download(str(folder/"comparacion_visual_pantheralab.jpg"))
if segmentation_overlay is not None:
    files.download(str(folder/"segmentacion_animal.jpg"))

**Interpretación científica:** MegaDetector identifica posibles animales; LR-ASPP genera **segmentación semántica** de categorías comunes (como perro), pero no individualiza animales ni identifica jaguares; SpeciesNet sugiere taxonomía. Los tres pueden equivocarse. El mejor resultado se obtiene evaluando el conjunto en varias fotos reales, no eligiendo el mayor porcentaje.

**Próximo paso para jaguares:** evaluar un segmentador de propósito general (por ejemplo, SAM 2 con anotación de puntos o cajas) y construir un conjunto de fotos de cámara trampa con máscaras anotadas por expertos. **Nada de esto es video automático o una función instalada en Render/Vercel.**